In [1]:
import pandas as pd
from pathlib import Path
import sys

ROOT = Path().resolve().parents[1] 
sys.path.append(str(ROOT))

from data_gathering.telpalen.telpalen_aantal_fietsers import haal_all_counts
from DWH.connection.connect import get_engine, getData
from data_gathering.telpalen.telpalen_locatie_fetcher import haal_telpalen_volledig_op
from data_gathering.hulp_functies import find_gemeente_in_location, get_nominatim_data

In [2]:
engine = get_engine()
df_telpalen = haal_telpalen_volledig_op(engine)
dim_location = getData(engine, query="SELECT * FROM DimLocation")

Reeds aanwezig in DimCountingPoint: 356 rijen


In [3]:
# Hernoemen kolommen
df_telpalen = df_telpalen.rename(columns={
    'counting_point_id' : 'CountingPointID',
    'customId' : 'CustomID',
    'name' : 'CountingPointName',
    'latitude' : 'Latitude',
    'longitude' :  'Longitude',
    'firstData' : 'FirstData',
    'lastData' : 'LastData',
    'granularity' : 'Granularity',
    'directional' : 'Directional',
    'direction_name_in' : 'DirectionNameIn',
    'direction_name_out' : 'DirectionNameOut',
    'domain_id' : 'DomainID',
    'domain_name' : 'DomainName',
    'description' : 'Description'
})

In [4]:
print(df_telpalen.info())

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
None


In [5]:
for index, row in df_telpalen.iterrows():
    naam = str(row['CountingPointName'])
    lat, lon = row['Latitude'], row['Longitude']
    pc = row['postcode']

    match = pd.DataFrame()

    if pc:
        try:
            match = dim_location[dim_location['PostalCode'] == pc]
        except (ValueError, AttributeError):
            match = pd.DataFrame()

    if len(match) == 1:
        municipality = str(match.iloc[0]['Municipality'])
    else:
        pc, municipality = get_nominatim_data(lat, lon)
    
    try:
        matchLocation = dim_location[(dim_location['PostalCode'] == pc) & (dim_location['Municipality'] == municipality.title())]
    except:
        print(f"pc = {pc}, municipality = {municipality}, index = {index}")
        print(f"PROBLEEM: [{index + 1}/{len(df_telpalen)}] Postcode = {pc}, loc = {naam}, mun = {municipality}, lat = {lat}, lon = {lon}")
        continue
    
    # als postcode en municipality niet matchen, zoek dan enkel op municipality
    if matchLocation.empty:
        matchLocation = dim_location[dim_location['Municipality'] == municipality.title()]

    # Geen match op municipality, zoek juiste Municipality met locatie naam
    if matchLocation.empty:
            municipality = find_gemeente_in_location(naam, locaties=match['Municipality'].tolist())
            matchLocation = dim_location[(dim_location['PostalCode'] == pc) & (dim_location['Municipality'] == municipality.title())]
    try:
        location_key = matchLocation.iloc[0].LocationKey
        df_telpalen.at[index,'LocationKey'] = location_key
        # print(f"[{index + 1}/{len(df_bluebike)}] Postcode = {pc}, loc = {row['LocationName']}, mun = {municipality}")
    except:
        print(f"PROBLEEM: [{index + 1}/{len(df_telpalen)}] Postcode = {pc}, loc = {naam}, mun = {municipality}, lat = {lat}, lon = {lon}")
    


In [6]:
df_telpalen

""


In [7]:
# dim_countingpoint = df_telpalen[[
#     "CountingPointID",
#     "CustomID",
#     "CountingPointName",
#     "Latitude",
#     "Longitude",
#     "FirstData",
#     "LastData",
#     "Granularity",
#     "Directional",
#     "DirectionNameIn",
#     "DirectionNameOut",
#     "DomainID",
#     "DomainName",
#     "Description",
#     "LocationKey"
# ]]
# dim_countingpoint["CountingPointID"] = dim_countingpoint["CountingPointID"].astype(int)
# dim_countingpoint["Latitude"] = dim_countingpoint["Latitude"].astype(float)
# dim_countingpoint["Longitude"] = dim_countingpoint["Longitude"].astype(float)
# dim_countingpoint["LocationKey"] = dim_countingpoint["LocationKey"].astype(int)

dim_countingpoint = getData(engine, query="SELECT * FROM DimCountingPoint")

dim_countingpoint.info()

<class 'pandas.DataFrame'>
RangeIndex: 356 entries, 0 to 355
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CountingPointID    356 non-null    int64  
 1   CustomID           256 non-null    str    
 2   CountingPointName  356 non-null    str    
 3   Latitude           356 non-null    float64
 4   Longitude          356 non-null    float64
 5   FirstData          356 non-null    object 
 6   Granularity        356 non-null    str    
 7   Directional        356 non-null    bool   
 8   DirectionNameIn    349 non-null    str    
 9   DirectionNameOut   337 non-null    str    
 10  DomainID           356 non-null    int64  
 11  DomainName         356 non-null    str    
 12  Description        116 non-null    str    
 13  LocationKey        356 non-null    int64  
dtypes: bool(1), float64(2), int64(3), object(1), str(7)
memory usage: 36.6+ KB


In [8]:
df_counts = haal_all_counts(dim_countingpoint, engine)

[1/356] ophalen data voor paal 100038932
Succes: Delete uitgevoerd
[2/356] ophalen data voor paal 100044065
Geen nieuwe data.
[3/356] ophalen data voor paal 100044847
Succes: Delete uitgevoerd
[4/356] ophalen data voor paal 100044848
Succes: Delete uitgevoerd
[5/356] ophalen data voor paal 100044849
Succes: Delete uitgevoerd
[6/356] ophalen data voor paal 100044850
Succes: Delete uitgevoerd
[7/356] ophalen data voor paal 100044851
Succes: Delete uitgevoerd
[8/356] ophalen data voor paal 100044852
Succes: Delete uitgevoerd
[9/356] ophalen data voor paal 100044853
Succes: Delete uitgevoerd
[10/356] ophalen data voor paal 100046096
Succes: Delete uitgevoerd
[11/356] ophalen data voor paal 100047495
Succes: Delete uitgevoerd
[12/356] ophalen data voor paal 100047496
Succes: Delete uitgevoerd
[13/356] ophalen data voor paal 100048586
Succes: Delete uitgevoerd
[14/356] ophalen data voor paal 100048587
Succes: Delete uitgevoerd
[15/356] ophalen data voor paal 100048588
Succes: Delete uitgevoe

In [13]:
df_counts["DateKey"] = df_counts["Date"].dt.strftime("%Y%m%d").astype(int)
df_counts

,CountingPointID,Date,DirectionInCounts,DirectionOutCounts,TotalCounts,DateKey
0,100038932,2026-01-01,602,657,1259,20260101
1,100038932,2026-01-02,1234,1028,2262,20260102
2,100038932,2026-01-03,871,855,1726,20260103
3,100038932,2026-01-04,641,674,1315,20260104
4,100038932,2026-01-05,1868,1767,3635,20260105
...,...,...,...,...,...,...
25969,300064248,2026-03-14,66,61,127,20260314
25970,300064248,2026-03-15,126,122,248,20260315
25971,300064248,2026-03-16,104,103,207,20260316
25972,300064248,2026-03-17,125,146,271,20260317


In [14]:
missing = 356 - len(df_counts["CountingPointID"].unique())
print(missing)

14


In [11]:
fact_counts = df_counts[[
    "CountingPointID",
    "DateKey",
    "DirectionInCounts",
    "DirectionOutCounts",
    "TotalCounts"
]]

In [12]:
print(fact_counts.info())
print(dim_countingpoint.info())

<class 'pandas.DataFrame'>
RangeIndex: 25974 entries, 0 to 25973
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   CountingPointID     25974 non-null  int64
 1   DateKey             25974 non-null  int64
 2   DirectionInCounts   25974 non-null  int64
 3   DirectionOutCounts  25974 non-null  int64
 4   TotalCounts         25974 non-null  int64
dtypes: int64(5)
memory usage: 1014.7 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 356 entries, 0 to 355
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CountingPointID    356 non-null    int64  
 1   CustomID           256 non-null    str    
 2   CountingPointName  356 non-null    str    
 3   Latitude           356 non-null    float64
 4   Longitude          356 non-null    float64
 5   FirstData          356 non-null    object 
 6   Granularity        356 non-null    str    
 7